# 02 - Cleaning and feature construction

**Stage 2 of 4.** Reads the raw ATP files in `../data`, cleans them, builds the target and the
features, verifies that no feature uses future information, and writes the modelling table to
`../data/processed/matches_model.csv.gz`.

Rules followed (instructor's note, 1 Nov 2026):

* every player feature (days since last match, form, surface record, head-to-head) is computed
  only from matches played **before** the match to predict;
* train/test splits are made by **date** (done in notebook 03).

Run from the folder `code/` in a fresh kernel. Shared functions are in `helpers.py`.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

import helpers as h

DATA_DIR = Path("../data")
OUT_DIR = DATA_DIR / "processed"
OUT_DIR.mkdir(exist_ok=True)

## 1. Load and sort chronologically

In [2]:
raw = h.load_raw(DATA_DIR)
prep = h.prepare(raw)
print(f"{len(raw):,} raw rows, {raw['tourney_date'].min()} to {raw['tourney_date'].max()}")
prep[["tourney_date", "round", "round_order", "approx_date", "played", "retired", "finished"]].head()

74,906 raw rows, 20000103 to 20241218


,tourney_date,round,round_order,approx_date,played,retired,finished
0,2000-01-03,R32,3,2000-01-05,True,False,True
1,2000-01-03,R32,3,2000-01-05,True,False,True
2,2000-01-03,R32,3,2000-01-05,True,False,True
3,2000-01-03,R32,3,2000-01-05,True,False,True
4,2000-01-03,R32,3,2000-01-05,True,False,True


## 2. Cleaning log

A match enters the **history** of both players if it was played (retirements included, since the
players did take the court). It enters the **modelling table** only if it was finished, both
players have a ranking, and the two ranks differ (otherwise there is no favourite).

In [3]:
n_raw = len(prep)
n_not_played = int((~prep["played"]).sum())
n_retired = int(prep["retired"].sum())
fin = prep[prep["finished"]]
n_missing_rank = int((fin["winner_rank"].isna() | fin["loser_rank"].isna()).sum())
fin_rk = fin[fin["winner_rank"].notna() & fin["loser_rank"].notna()]
n_equal_rank = int((fin_rk["winner_rank"] == fin_rk["loser_rank"]).sum())

featured = h.add_history_features(prep)
table = h.build_model_table(featured)

log = pd.DataFrame({
    "step": ["raw rows",
             "removed: not played (walkover, default, abandoned)",
             "removed: retirements (RET)",
             "removed: missing ranking of at least one player",
             "removed: equal ranks (no favourite)",
             "modelling table"],
    "matches": [n_raw, -n_not_played, -n_retired, -n_missing_rank, -n_equal_rank, len(table)],
})
assert n_raw - n_not_played - n_retired - n_missing_rank - n_equal_rank == len(table)
log

,step,matches
0,raw rows,74906
1,"removed: not played (walkover, default, abando...",-419
2,removed: retirements (RET),-2223
3,removed: missing ranking of at least one player,-1692
4,removed: equal ranks (no favourite),-8
5,modelling table,70564


## 3. Check that no feature uses the future

Two independent tests (see `helpers.check_no_lookahead`):

1. **Brute force.** For 300 random matches, both players' features are recomputed from scratch by
   filtering the data to strictly earlier matches, and compared with the table. The Elo ratings
   cannot be recomputed from one player's history alone (they depend on the opponents' ratings), so
   they are compared with a second, independent implementation of the same formulas.
2. **Truncation.** The data is cut right after a random match, all features are rebuilt, and the
   features of that match must be identical to the full run.

In [4]:
report = h.check_no_lookahead(prep, table, n_samples=300, n_truncations=5, seed=0)
print("all checks passed:", report)

all checks passed: {'brute_force_matches': 300, 'truncation_points': 5}


## 4. What the table looks like

`upset = 1` when the worse-ranked player wins. The naive baseline "the favourite always wins"
reaches `1 - upset rate` accuracy; any model has to beat it.

In [5]:
upset_rate = table["upset"].mean()
print(f"matches: {len(table):,}  |  upset rate: {upset_rate:.1%}  |  baseline accuracy: {1 - upset_rate:.1%}")
print()
print(table.groupby("year")["upset"].agg(["size", "mean"]).round(3).T.to_string())

matches: 70,564  |  upset rate: 34.1%  |  baseline accuracy: 65.9%

year      2000      2001      2002      2003      2004     2005      2006     2007      2008      2009      2010      2011      2012      2013      2014      2015      2016     2017      2018      2019      2020      2021     2022      2023     2024
size  3151.000  3208.000  2987.000  2898.000  3096.000  3091.00  3073.000  3093.00  2830.000  2876.000  2849.000  2814.000  2891.000  2743.000  2637.000  2767.000  2769.000  2748.00  2858.000  2601.000  1387.000  2622.000  2805.00  2843.000  2927.00
mean     0.356     0.366     0.366     0.346     0.352     0.33     0.334     0.34     0.321     0.315     0.324     0.316     0.317     0.328     0.313     0.308     0.319     0.34     0.358     0.387     0.357     0.363     0.35     0.363     0.36


In [6]:
feature_cols = ["log_rank_ratio", "log_points_ratio", "fav_days", "und_days", "fav_form", "und_form",
                "form_diff", "fav_surf", "und_surf", "surf_diff", "h2h_diff", "h2h_total",
                "elo_diff", "selo_diff"]
display(table[feature_cols].describe().round(2).T)
share_missing = table[feature_cols].isna().mean().round(3)
print("share missing (a player with no earlier match in the data):")
print(share_missing[share_missing > 0].to_string())

,count,mean,std,min,25%,50%,75%,max
log_rank_ratio,70564.0,1.17,0.95,0.00,0.44,0.94,1.67,7.22
log_points_ratio,70564.0,0.94,0.84,0.00,0.33,0.73,1.32,8.66
fav_days,70292.0,13.45,35.37,0.00,1.00,4.00,13.00,365.00
und_days,69038.0,26.97,68.01,0.00,1.00,2.00,14.00,365.00
fav_form,70292.0,0.56,0.19,0.00,0.40,0.60,0.70,1.00
und_form,69038.0,0.46,0.20,0.00,0.30,0.50,0.60,1.00
form_diff,68883.0,0.10,0.22,-1.00,0.00,0.10,0.20,1.00
fav_surf,70530.0,0.55,0.13,0.11,0.47,0.55,0.63,0.93
und_surf,70530.0,0.49,0.11,0.12,0.41,0.50,0.56,0.93
surf_diff,70530.0,0.06,0.14,-0.64,-0.03,0.06,0.16,0.68


share missing (a player with no earlier match in the data):
fav_days     0.004
und_days     0.022
fav_form     0.004
und_form     0.022
form_diff    0.024


Missing values appear only where a player has no earlier match in the dataset (the first
appearances, mostly in 2000-2001). They are not filled here: the imputation is done inside the
cross-validation in notebook 03, so that nothing is learned from test data.

**Definition note.** `days since last match` uses an approximate match date (tournament start plus
a number of days that grows with the round), so a favourite who plays a quarter-final the day after
the round of 16 gets about 1-2 days. This differs from the exploratory version used for the proposal,
which was based on the tournament start date alone, and explains why the median is lower here.

**Elo ratings.** Every player starts at 1500. After each match the winner gains and the loser
loses `K * (1 - expected)`, where `expected` is the win probability implied by the two ratings read
before the match and `K = 250 / (matches played + 5) ** 0.4` is large for newcomers and shrinks with
experience. A second rating per surface is kept in the same way. `elo_diff` and `selo_diff` are the
favourite's rating minus the underdog's. The first years of the data act as a burn-in: ratings from
2000-2002 are noisy because everybody starts at the same value. Retirements count in the Elo
history like in the other features.

**Limitation.** The data are tour-level only (no Challengers or Futures), so the history of a player
who dropped down a level has gaps and his inactivity is overstated.

## 5. Save

In [7]:
out = OUT_DIR / "matches_model.csv.gz"
table.to_csv(out, index=False)
print(out, f"{out.stat().st_size / 1e6:.1f} MB", table.shape)

../data/processed/matches_model.csv.gz 9.2 MB (70564, 40)
